## Simulating Future Data

In [1]:
import numpy as np
import pandas as pd

def simulate_heston(
    s0=75,
    v0=0.04,
    mu=0.05,
    kappa=2.0,
    theta=0.04,
    xi=0.4,
    rho=-0.6,
    n=252,
    dt=1/252,
    iv0=0.25,
    iv_kappa=5.0,
    iv_noise=0.05,
    seed=1156
):
    np.random.seed(seed)

    oil_price = np.zeros(n)
    oil_vol = np.zeros(n)
    v = np.zeros(n)

    oil_price[0] = s0
    v[0] = v0
    oil_vol[0] = iv0

    for t in range(1, n):
        z1 = np.random.randn()
        z2 = rho * z1 + np.sqrt(1 - rho**2) * np.random.randn()
        z3 = np.random.randn()

        v_prev = max(v[t-1], 0)

        v[t] = v[t-1] + kappa * (theta - v_prev) * dt + xi * np.sqrt(v_prev * dt) * z2
        v[t] = max(v[t], 0)

        oil_price[t] = oil_price[t-1] * np.exp(
            (mu - 0.5 * v_prev) * dt + np.sqrt(v_prev * dt) * z1
        )

        true_vol = np.sqrt(v[t])

        oil_vol[t] = oil_vol[t-1] + iv_kappa * (true_vol - oil_vol[t-1]) * dt \
                     + iv_noise * np.sqrt(dt) * z3
        oil_vol[t] = max(oil_vol[t], 0.01)

    df = pd.DataFrame({
        "Oil_Price": oil_price,
        "Oil_Vol": oil_vol
    })

    df.columns.name = "Ticker"

    return df
#I will assume S0 will be most recent closing value- but could be whatever
S0 = 92.13

#Pick regime from below (We can add more or only use 1 if we want )

# Calm range-bound
test_data = simulate_heston(
    s0=S0, n=1008, v0=0.06, mu=0.02, kappa=3.0, theta=0.05,
    xi=0.35, rho=-0.25, iv0=0.28, iv_kappa=6.0, iv_noise=0.04, seed=1
)

# # Moderate bullish
# future_data = simulate_heston(
#     s0=S0, n=504, v0=0.07, mu=0.08, kappa=2.5, theta=0.06,
#     xi=0.45, rho=-0.35, iv0=0.30, iv_kappa=6.0, iv_noise=0.04, seed=2
# )

# # Stress selloff
# future_data = simulate_heston(
#     s0=S0, n=504, v0=0.12, mu=-0.12, kappa=1.5, theta=0.12,
#     xi=0.70, rho=-0.55, iv0=0.40, iv_kappa=6.0, iv_noise=0.04, seed=3
# )

#Making this constant, think simulating is unnecessary
test_data['rf'] = 0.04

## Getting Training Data

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm
from arch import arch_model


oil_price = yf.download("CL=F", start="2005-01-01", end="2026-04-22")['Close']
oil_vol = yf.download("^OVX", start="2005-01-01", end="2026-04-22")['Close']

oil_price = oil_price.rename(columns={'CL=F': 'Oil_Price'})
oil_vol = oil_vol.rename(columns={'^OVX': 'Oil_Vol'})

train_df = pd.merge(oil_price, oil_vol, left_index=True, right_index=True)

train_df['Oil_Price'] = train_df['Oil_Price'].astype(float)
train_df['Oil_Vol'] = train_df['Oil_Vol'].astype(float) / 100

train_df['Oil_Price'] = train_df['Oil_Price'].mask(train_df['Oil_Price'] < 0, np.nan).ffill()

train_df = train_df.dropna().reset_index(drop=True)

/var/folders/_q/9tmf12095cn84735n3vm06sm0000gn/T/ipykernel_17375/522350641.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  oil_price = yf.download("CL=F", start="2005-01-01", end="2026-04-22")['Close']
[*********************100%***********************]  1 of 1 completed
/var/folders/_q/9tmf12095cn84735n3vm06sm0000gn/T/ipykernel_17375/522350641.py:9: FutureWarning: YF.download() has changed argument auto_adjust default to True
  oil_vol = yf.download("^OVX", start="2005-01-01", end="2026-04-22")['Close']
[*********************100%***********************]  1 of 1 completed


## Testing Strategy on Simulated Data

In [3]:
def call_price(F, K, r, sigma, T=30/365):
    d1 = (np.log(F/K) + 0.5*sigma**2*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return np.exp(-r*T) * (F*norm.cdf(d1) - K*norm.cdf(d2))

def put_price(F, K, r, sigma, T=30/365):
    d1 = (np.log(F/K) + 0.5*sigma**2*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    return np.exp(-r*T) * (K*norm.cdf(-d2) - F*norm.cdf(-d1))

def test_vol_strategy(train_df, test_df, hold_days=30, z_window=150,
                      upper=1.75, lower=-2, rf=0.04, tcost=0.00):

    hist = train_df[['Oil_Price', 'Oil_Vol']].copy()
    fut = test_df[['Oil_Price', 'Oil_Vol']].copy()

    full = pd.concat([hist, fut], ignore_index=True)

    returns = np.log(full['Oil_Price'] / full['Oil_Price'].shift(1)).dropna() * 100
    split_idx = len(hist) - 1

    model = arch_model(returns, mean='Constant', vol='Garch', p=1, q=1, dist='t')
    fit = model.fit(last_obs=split_idx, disp='off')

    fcst = fit.forecast(horizon=hold_days, start=split_idx)
    garch_vol = np.sqrt(fcst.variance[f'h.{hold_days}']) / 100 * np.sqrt(252)

    full['Forecasted_Vol'] = garch_vol.reindex(full.index)
    full['Diff'] = (full['Oil_Vol'] - full['Forecasted_Vol'])
    full['Z_Score'] = (full['Diff'] - full['Diff'].rolling(z_window).mean()) / full['Diff'].rolling(z_window).std()

    df = full.iloc[len(hist):].copy().reset_index(drop=True)

    df['Strike'] = (df['Oil_Price'] * 2).round() / 2
    df['Call'] = call_price(df['Oil_Price'], df['Strike'], rf, df['Oil_Vol'])
    df['Put'] = put_price(df['Oil_Price'], df['Strike'], rf, df['Oil_Vol'])
    df['Straddle'] = df['Call'] + df['Put']

    df['Future_Price'] = df['Oil_Price'].shift(-hold_days)
    df['Realized_Move'] = abs(df['Future_Price'] - df['Strike'])

    df['Signal'] = 0
    #df.loc[df['Z_Score'] < lower, 'Signal'] = 1
    df.loc[df['Z_Score'] > upper, 'Signal'] = 1

    df['PnL'] = 0.0
    df.loc[df['Signal'] == 1, 'PnL'] = df['Realized_Move'] - df['Straddle']
    df.loc[df['Signal'] == -1, 'PnL'] = df['Straddle'] - df['Realized_Move']

    df['PnL'] -= abs(df['Signal']) * tcost * df['Straddle']
    df['Trade_Return'] = df['PnL'] / df['Straddle']

    trades = df[
        (df['Signal'] != 0) &
        (df['Straddle'] > 0) &
        (~df['Future_Price'].isna())
    ].copy()

    rets = trades['Trade_Return'].dropna()
    trades_per_year = 252/30

    stats = pd.Series({
        'num_trades': len(trades),
        'win_rate': (rets > 0).mean(),
        'mean_return': rets.mean(),
        'annualized_return': rets.mean() * trades_per_year,
        'std_return': rets.std(),
        'annualized_std': rets.std() * np.sqrt(trades_per_year),
        'sharpe': np.sqrt(trades_per_year) * rets.mean() / rets.std() if rets.std() > 0 else np.nan,
        'total_return_uncompounded': rets.sum()
    })

    return df, trades, stats
results_df, trades_df, stats = test_vol_strategy(train_df, test_data)
stats

num_trades                   69.000000
win_rate                      0.666667
mean_return                   0.576952
annualized_return             4.846395
std_return                    1.102362
annualized_std                3.194948
sharpe                        1.516893
total_return_uncompounded    39.809677
dtype: float64